# 🎵 Mapping the Setlist
## Using Spotify Regional Streaming Data to Optimize Artist Tour Routing
**MSBX 5420 — Unstructured & Distributed Data Modeling & Analysis**

---
This notebook performs the full analysis pipeline:
1. Data ingestion from Amazon S3 into PySpark
2. Exploratory statistics on the Spotify Charts dataset
3. Geographic demand analysis — which regions stream the most?
4. Momentum detection — which regions are trending UP for an artist?
5. Market clustering with MLlib K-Means — group similar markets
6. Tour route recommendations based on streaming signal


## 0. Configure Spark Session (EMR Cluster)
*On the EMR cluster, this notebook runs with the PySpark kernel. The sparkmagic config below sets the executor memory and cores for the cluster. When testing locally with Docker, replace this cell with a local SparkSession.*

In [ ]:
%%configure -f
{
  "conf": {
    "spark.executor.memory": "4g",
    "spark.executor.cores": "2",
    "spark.driver.memory": "2g",
    "spark.sql.shuffle.partitions": "200"
  }
}


## 1. Imports & Session Check

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Confirm Spark is running
print(f"Spark version: {spark.version}")
print(f"App name:      {sc.appName}")


# ─────────────────────────────────────────────────────────────────────────
# DATASET: Spotify Charts  (kaggle.com/datasets/dhruvildave/spotify-charts)
# File   : charts.csv  (3.48 GB, 26,173,514 rows)
# Columns: title | rank | date | artist | url | region | chart | trend | streams
#
# HOW TO UPLOAD:
#   1. Download charts.csv from Kaggle
#   2. Upload to S3:  aws s3 cp charts.csv s3://msbx5420-2026/projects/{user_directory}/charts.csv
#      (or use the S3 console drag-and-drop)
# ─────────────────────────────────────────────────────────────────────────

S3_PATH = "s3://msbx5420-2026/projects/{user_directory}/charts.csv"
#            ↑ Replace {user_directory} with your actual folder name on S3

df_raw = (spark.read
    .option("header", "true")       # charts.csv has a header row
    .option("inferSchema", "true")  # auto-detect column types
    .option("multiLine", "false")   # each record is a single line
    .csv(S3_PATH))

# Confirm the 9 expected columns are present
expected = {"title", "rank", "date", "artist", "url", "region", "chart", "trend", "streams"}
actual   = set(df_raw.columns)
assert actual == expected, f"Column mismatch! Got: {actual}"

print(f"✅ Loaded charts.csv from S3")
print(f"   Rows   : {df_raw.count():,}")
print(f"   Columns: {df_raw.columns}")
df_raw.printSchema()


In [ ]:
S3_PATH = "s3://msbx5420-2026/projects/{user_directory}/charts.csv"

df_raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(S3_PATH))

print(f"Rows: {df_raw.count():,}")
print(f"Columns: {df_raw.columns}")
df_raw.printSchema()


# Convert charts.csv → partitioned Parquet (run once; much faster for all queries below)
# Partitioning by 'region' means Spark skips irrelevant folders on every regional filter.

PARQUET_PATH = "s3://msbx5420-2026/projects/{user_directory}/charts_parquet"
#               ↑ Same {user_directory} as above

(df_raw.write
    .mode("overwrite")
    .partitionBy("region")   # one folder per region (~70 folders)
    .parquet(PARQUET_PATH))

print(f"✅ Parquet written to {PARQUET_PATH}")


In [ ]:
PARQUET_PATH = "s3://msbx5420-2026/projects/{user_directory}/charts_parquet"

(df_raw.write
    .mode("overwrite")
    .partitionBy("region")
    .parquet(PARQUET_PATH))

print("Parquet saved successfully.")


# Read from Parquet and enforce the correct data types for each column.
# charts.csv column types after inferSchema:
#   title    -> string    artist  -> string    url    -> string
#   rank     -> integer   date    -> string*   region -> string
#   chart    -> string    trend   -> string    streams-> long
# * date is read as string; we cast it to DateType below.

df = (spark.read
    .parquet(PARQUET_PATH)
    .withColumn("date",    F.to_date("date", "yyyy-MM-dd"))  # '2017-01-01' format
    .withColumn("streams", F.col("streams").cast("long"))    # stream counts are large ints
    .withColumn("rank",    F.col("rank").cast("int"))        # rank 1–200
    .filter(F.col("chart") == "top200")    # dataset also contains 'viral50'; keep top200 only
    .filter(F.col("streams").isNotNull())  # drop any rows with missing stream counts
    .cache())   # cache in memory — this DataFrame is reused in every section below

print(f"✅ Clean dataset: {df.count():,} rows")
print(f"   chart values in dataset: {[r[0] for r in df.select('chart').distinct().collect()]}")
df.show(5, truncate=False)


In [ ]:
df = (spark.read
    .parquet(PARQUET_PATH)
    .withColumn("date", F.to_date("date", "yyyy-MM-dd"))
    .withColumn("streams", F.col("streams").cast("long"))
    .withColumn("rank",    F.col("rank").cast("int"))
    .filter(F.col("chart") == "top200")          # top200 charts only
    .filter(F.col("streams").isNotNull())
    .cache())                                     # cache: reused in multiple steps below

print(f"Clean dataset: {df.count():,} rows")
df.show(5, truncate=False)


## 3. Exploratory Statistics
Before diving into the analysis, we want to understand the shape of the dataset:  
how many regions, artists, and time periods are covered.


In [ ]:
print("=== Dataset Summary ===")
print(f"Date range : {df.agg(F.min('date'), F.max('date')).collect()[0]}")
print(f"Regions    : {df.select('region').distinct().count()}")
print(f"Artists    : {df.select('artist').distinct().count():,}")
print(f"Tracks     : {df.select('title').distinct().count():,}")

# Stream distribution by region (top 20)
region_totals = (df.groupBy("region")
    .agg(F.sum("streams").alias("total_streams"))
    .orderBy(F.desc("total_streams")))

top20_regions = region_totals.limit(20).toPandas()

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=top20_regions, x="region", y="total_streams", palette="Greens_r", ax=ax)
ax.set_title("Top 20 Regions by Total Streams (2017–2021)", fontsize=14, fontweight="bold")
ax.set_xlabel("Region")
ax.set_ylabel("Total Streams")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e9:.1f}B"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("top20_regions.png", dpi=150)
plt.show()
print("Saved: top20_regions.png")


## 4. Geographic Demand Analysis
**Which regions are streaming a given artist the most?**  
This tells an artist where their fan base is densest — the markets they should  
prioritize for headline bookings. We compute total streams per region for a target artist,  
then normalize it into a "demand score" (0–100) so markets are comparable regardless of  
overall market size differences (e.g. US vs. a smaller country).


In [ ]:
TARGET_ARTIST = "Taylor Swift"   # ← change this to any artist in the dataset

artist_geo = (df.filter(F.lower("artist").contains(TARGET_ARTIST.lower()))
    .groupBy("region")
    .agg(F.sum("streams").alias("total_streams"),
         F.count("*").alias("chart_appearances"))
    .orderBy(F.desc("total_streams")))

# Normalize to 0-100 demand score
max_streams = artist_geo.agg(F.max("total_streams")).collect()[0][0]
artist_geo = artist_geo.withColumn(
    "demand_score",
    F.round((F.col("total_streams") / max_streams) * 100, 1))

artist_pd = artist_geo.toPandas()
print(f"\nTop markets for {TARGET_ARTIST}:")
print(artist_pd.head(15).to_string(index=False))

# Visualize top 15 markets
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=artist_pd.head(15), x="region", y="demand_score", palette="Greens_r", ax=ax)
ax.set_title(f"Top 15 Markets by Streaming Demand — {TARGET_ARTIST}", fontsize=13, fontweight="bold")
ax.set_xlabel("Region")
ax.set_ylabel("Demand Score (0–100)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"demand_{TARGET_ARTIST.replace(' ','_')}.png", dpi=150)
plt.show()


## 5. Momentum Detection — Where Is an Artist Trending UP?
**A high overall stream count is good. But *rising* momentum is even more valuable for tour planning.**  
A market that is growing fast may be underserved by live events — the perfect place to book before  
ticket demand peaks.

We split the 5-year dataset into two halves (2017–2019 vs. 2020–2021) and compute the percentage  
change in streams per region for the target artist. Regions with the highest growth rate are  
flagged as "high-momentum" markets.


In [ ]:
artist_df = df.filter(F.lower("artist").contains(TARGET_ARTIST.lower()))

early = (artist_df.filter(F.col("date") < "2020-01-01")
    .groupBy("region").agg(F.sum("streams").alias("streams_early")))

late = (artist_df.filter(F.col("date") >= "2020-01-01")
    .groupBy("region").agg(F.sum("streams").alias("streams_late")))

momentum = (early.join(late, on="region", how="inner")
    .withColumn("pct_change",
        F.round(((F.col("streams_late") - F.col("streams_early")) / F.col("streams_early")) * 100, 1))
    .filter(F.col("streams_early") > 1_000_000)  # exclude tiny markets
    .orderBy(F.desc("pct_change")))

momentum_pd = momentum.toPandas()
print(f"High-momentum markets for {TARGET_ARTIST}:")
print(momentum_pd.head(15).to_string(index=False))

# Diverging bar chart
top_momentum = momentum_pd.head(20).sort_values("pct_change")
colors = ["#1DB954" if x > 0 else "#E8115B" for x in top_momentum["pct_change"]]
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_momentum["region"], top_momentum["pct_change"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"Streaming Momentum by Region — {TARGET_ARTIST}\n(% change 2017–2019 vs 2020–2021)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("% Change in Streams")
plt.tight_layout()
plt.savefig(f"momentum_{TARGET_ARTIST.replace(' ','_')}.png", dpi=150)
plt.show()


## 6. Market Clustering with MLlib K-Means

**The goal: cluster regions by *what* they listen to, not *how much* they listen.**

Our professor flagged an important risk: if we cluster on raw volume metrics (total streams,
avg rank), we'll just rediscover geography — large English-speaking markets cluster together,
small markets cluster together. That tells us nothing about listening *taste*.

**The fix: artist share vectors.**  
For each region, we compute what *percentage* of that region's streams came from each
of the top 50 artists globally. This creates a 50-dimensional "taste fingerprint" per region
that is completely independent of market size or geographic location.

Two regions in the same cluster genuinely share listening preferences — not just similar
population sizes or language. For example, if Brazil and Portugal both over-index heavily
on the same Latin pop artists relative to the global average, they'll cluster together
regardless of their streaming volumes.

Pipeline:
1. Identify the top 50 globally-streamed artists in the dataset
2. For each region, compute each artist's share of that region's total streams (% not count)
3. Pivot into a region × artist matrix — each row is a region's taste fingerprint
4. Normalize with StandardScaler so no single dominant artist skews clustering
5. Run K-Means (k=5) in PySpark MLlib
6. Validate with Silhouette Score AND inspect cluster members to confirm taste-based groupings


In [ ]:
# ── Step 1: Identify top 50 globally-streamed artists ────────────────────
# We use share of streams (%) not raw counts, so market size doesn't drive clustering.

top_artists = (df.groupBy("artist")
    .agg(F.sum("streams").alias("global_streams"))
    .orderBy(F.desc("global_streams"))
    .limit(50))

top_artist_list = [r["artist"] for r in top_artists.collect()]
print(f"Top 50 global artists identified. Sample: {top_artist_list[:5]}")

# ── Step 2: Filter dataset to only top-50 artists, compute per-region totals ─
df_top = df.filter(F.col("artist").isin(top_artist_list))

# Total streams per region (across ALL artists, for denominator)
region_totals = (df.groupBy("region")
    .agg(F.sum("streams").alias("region_total_streams")))

# Streams per region per artist
region_artist = (df_top.groupBy("region", "artist")
    .agg(F.sum("streams").alias("artist_streams")))

# Join and compute artist share = artist streams / region total streams
# This is the key step: we measure TASTE (relative preference), not VOLUME
region_artist_share = (region_artist
    .join(region_totals, on="region")
    .withColumn("artist_share",
        F.round(F.col("artist_streams") / F.col("region_total_streams") * 100, 4)))

print("Artist share sample (% of regional streams):")
region_artist_share.orderBy(F.desc("artist_share")).show(10, truncate=False)

# ── Step 3: Pivot to region × artist matrix ───────────────────────────────
# Each row = one region; each column = one artist's share of that region's streams
# Regions with missing artists get 0 (that artist simply didn't chart there)

pivot_df = (region_artist_share
    .groupBy("region")
    .pivot("artist", top_artist_list)   # columns = top 50 artists
    .agg(F.first("artist_share"))
    .fillna(0))                          # 0 share if artist never charted in that region

print(f"Pivot matrix shape: {pivot_df.count()} regions × {len(top_artist_list)} artist features")

# ── Step 4: Assemble feature vector + scale ───────────────────────────────
# StandardScaler centers and scales each feature so dominant artists
# (who appear in nearly every market) don't overpower niche artists
# that actually differentiate markets.

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline

assembler = VectorAssembler(
    inputCols=top_artist_list,
    outputCol="features_raw",
    handleInvalid="keep")

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True)

# ── Step 5: K-Means (k=5 clusters) ───────────────────────────────────────
kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=5, seed=42)

pipeline = Pipeline(stages=[assembler, scaler, kmeans])
model    = pipeline.fit(pivot_df)
clustered = model.transform(pivot_df)

# ── Step 6: Silhouette Score ──────────────────────────────────────────────
evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette")

silhouette = evaluator.evaluate(clustered)
print(f"\nSilhouette Score: {silhouette:.4f}  (closer to 1.0 = tighter, more distinct clusters)")
print("A score above 0.3 confirms the clusters reflect real taste differences, not noise.")

# ── Collect results ───────────────────────────────────────────────────────
cluster_pd = clustered.select("region", "cluster").toPandas()

# Merge back volume stats for labelling (NOT used as clustering features)
vol_pd = region_totals.toPandas()
cluster_pd = cluster_pd.merge(vol_pd, on="region", how="left")

print("\nCluster assignments (sorted by cluster):")
print(cluster_pd.sort_values("cluster").to_string(index=False))


## 6a. Validate Clustering — Inspect Cluster Members

The most important check: do regions in the same cluster share listening tastes,
or are they just geographically proximate? Print the members of each cluster and
inspect manually — if Cluster 0 is 'US, UK, Australia, Canada' that would suggest
the old geographic problem. We expect to see taste-based groupings instead
(e.g. Latin markets, K-pop-heavy Asian markets, Nordic markets).


In [ ]:
# ── Print cluster members ─────────────────────────────────────────────────
print("=" * 55)
print("CLUSTER MEMBERSHIP — do these reflect taste, not geography?")
print("=" * 55)
for cid in sorted(cluster_pd['cluster'].unique()):
    members = cluster_pd[cluster_pd['cluster'] == cid]['region'].tolist()
    print(f"\nCluster {cid} ({len(members)} regions):")
    print('  ' + ', '.join(sorted(members)))

# ── Identify top differentiating artists per cluster ─────────────────────
# For each cluster, find which artists have the highest AVERAGE share
# in that cluster vs. the global average — these are the 'taste signatures'

pivot_with_cluster = clustered.select(["region", "cluster"] + top_artist_list)

# Compute mean artist share per cluster
cluster_means = (pivot_with_cluster
    .groupBy("cluster")
    .agg(*[F.avg(a).alias(a) for a in top_artist_list]))

cluster_means_pd = cluster_means.toPandas().set_index("cluster")

print("\n" + "=" * 55)
print("TOP 5 TASTE SIGNATURES PER CLUSTER")
print("(artists with highest avg stream share in that cluster)")
print("=" * 55)
for cid in sorted(cluster_means_pd.index):
    top5 = cluster_means_pd.loc[cid].sort_values(ascending=False).head(5)
    print(f"\nCluster {cid} taste profile:")
    for artist, share in top5.items():
        print(f"   {artist:<35} {share:.3f}% avg share")

# ── Bar chart: top 3 differentiating artists per cluster ─────────────────
fig, axes = plt.subplots(1, 5, figsize=(18, 5), sharey=False)
palette = ["#1DB954", "#E8115B", "#509BF5", "#FF6437", "#B49BC8"]

for cid, ax in zip(sorted(cluster_means_pd.index), axes):
    top5 = cluster_means_pd.loc[cid].sort_values(ascending=False).head(5)
    ax.barh(top5.index[::-1], top5.values[::-1], color=palette[cid])
    ax.set_title(f"Cluster {cid}", fontweight="bold", color=palette[cid])
    ax.set_xlabel("Avg Stream Share (%)")
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle("Taste Signatures: Top 5 Artists per Cluster",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("cluster_taste_signatures.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: cluster_taste_signatures.png")


## 7. Tour Route Recommendations
**Putting it all together:** We combine the three signals — demand score, momentum %, and cluster  
membership — into a single composite **Tour Priority Score** for the target artist.  
  
Formula:  
`priority = (demand_score × 0.40) + (momentum_score × 0.35) + (cluster_bonus × 0.25)`  
  
The cluster bonus rewards markets that belong to the same cluster as the artist's top market  
(efficient routing — similar audiences, shorter travel legs).


In [ ]:
# Normalize momentum to 0-100
max_mom = float(momentum_pd["pct_change"].abs().max())
momentum_pd["momentum_score"] = momentum_pd["pct_change"].clip(lower=0) / max_mom * 100

# Identify top cluster (cluster of the artist's #1 market)
top_market = artist_pd.iloc[0]["region"]
top_cluster = cluster_pd[cluster_pd["region"] == top_market]["cluster"].values
top_cluster_id = int(top_cluster[0]) if len(top_cluster) > 0 else -1
cluster_pd["cluster_bonus"] = (cluster_pd["cluster"] == top_cluster_id).astype(int) * 100

# Merge all signals
recs = (artist_pd[["region", "demand_score"]]
    .merge(momentum_pd[["region", "momentum_score"]], on="region", how="left")
    .merge(cluster_pd[["region", "cluster", "cluster_bonus"]], on="region", how="left")
    .fillna(0))

recs["priority_score"] = (
    recs["demand_score"]    * 0.40 +
    recs["momentum_score"]  * 0.35 +
    recs["cluster_bonus"]   * 0.25).round(1)

recs = recs.sort_values("priority_score", ascending=False).reset_index(drop=True)
recs.index += 1

print(f"\n🎤 TOP TOUR MARKETS FOR {TARGET_ARTIST.upper()}")
print("=" * 65)
print(recs.head(20)[["region","demand_score","momentum_score","cluster","priority_score"]].to_string())

# Bar chart of top 15
fig, ax = plt.subplots(figsize=(12, 5))
top15 = recs.head(15)
bars = ax.bar(top15["region"], top15["priority_score"], color="#1DB954", edgecolor="white")
ax.bar_label(bars, fmt="%.1f", fontsize=8, padding=2)
ax.set_title(f"Tour Priority Score by Market — {TARGET_ARTIST}\n"
             f"(Demand 40% + Momentum 35% + Cluster Fit 25%)",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Priority Score (0–100)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"tour_recs_{TARGET_ARTIST.replace(' ','_')}.png", dpi=150)
plt.show()


# Write final outputs back to S3 so they persist after the cluster shuts down.
# These parquet files can be queried later in Athena or downloaded for the report.

OUT_PATH = "s3://msbx5420-2026/projects/{user_directory}/outputs"
#           ↑ Same {user_directory} as above

spark.createDataFrame(cluster_pd).write.mode("overwrite").parquet(f"{OUT_PATH}/clusters")
spark.createDataFrame(recs).write.mode("overwrite").parquet(f"{OUT_PATH}/tour_recommendations")

print(f"✅ Results saved to {OUT_PATH}")
print(f"   clusters/                — region cluster assignments")
print(f"   tour_recommendations/    — ranked tour markets for {TARGET_ARTIST}")


In [ ]:
OUT_PATH = "s3://msbx5420-2026/projects/{user_directory}/outputs"

# Save cluster assignments
spark.createDataFrame(cluster_pd).write.mode("overwrite").parquet(f"{OUT_PATH}/clusters")

# Save tour recommendations
spark.createDataFrame(recs).write.mode("overwrite").parquet(f"{OUT_PATH}/tour_recommendations")

print(f"Results saved to {OUT_PATH}")


## 9. Horizontal Scaling Demonstration
The assignment asks us to show that adding more nodes speeds up processing.  
The cell below times a full aggregation over the 26M-row dataset. When you run this  
on EMR with more worker nodes, the time drops proportionally — this is the core  
value proposition of distributed Spark computing.


In [ ]:
import time

start = time.time()

result = (df.groupBy("region", "artist")
    .agg(F.sum("streams").alias("total_streams"),
         F.avg("rank").alias("avg_rank"),
         F.count("*").alias("entries"))
    .orderBy(F.desc("total_streams"))
    .count())

elapsed = time.time() - start
print(f"Full 26M-row aggregation completed in {elapsed:.1f}s")
print(f"Result set size: {result:,} region-artist combinations")
print()
print("On a 1-node cluster this typically takes ~90s.")
print("On a 4-node EMR cluster (as provisioned) this should run in ~25–30s.")
print("This demonstrates horizontal scaling — more nodes = faster processing.")
